# **Modelo LightGCN**
### Proyecto Hito 3
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- Definición del modelo, formateo de datos, y entrenamiento](#2--definición-del-modelo-formateo-de-datos-y-entrenamiento)

>[3- Generación de recomendaciones](#3--generación-de-recomendaciones)

>[4- Métricas](#4--métricas)

>[5- Referencias](#5--referencias)

## 0- Instalación de librerías

En caso de usar Colab, correr estas celdas. Si se corre en local, se deben tener exactamente las mismas versiones de las librerías indicadas en estas celdas. Las versiones de las librerías son:

- numpy: 1.25.0
- pandas: 2.2.2
- scipy: 1.10.1
- tqdm: 4.66.6
- torch: 2.5.1+cpu
- recbole: 1.2.1

In [ ]:
# !pip uninstall -y numpy
# !pip install numpy==1.25

In [ ]:
# !pip uninstall -y pandas
# !pip install numpy==2.2.2

In [ ]:
# !pip uninstall -y scipy
# !pip install scipy==1.10.1

In [ ]:
# !pip uninstall -y tqdm
# !pip install tqdm==4.66.6

In [ ]:
# !pip uninstall -y torch
# !pip uninstall -y torchvision
# !pip uninstall -y torchaudio
# !pip install torch==2.5.1+cpu --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# !pip install recbole==1.2.1

## 1- Carga de datos

Se leen los archivos de datos (entrenamiento, testeo, y validación) y se almacenan en un dataframe:

In [1]:
import pandas as pd

df_train = pd.read_csv('train_split.csv')
df_test = pd.read_csv('test_split.csv')
df_val = pd.read_csv('val_split.csv')

La estructura de estos archivos es:

In [2]:
df_train.head(5)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,322330,0,0,2019-07-02,True,67.5,731,33484606
1,433340,0,0,2020-01-24,True,32.3,731,26236770
2,394360,2,0,2020-04-20,True,403.7,731,25992499
3,4700,0,0,2020-04-21,True,683.5,731,9845461
4,246090,3,0,2015-02-25,False,7.9,3128,28148695


En este dataframe se guarda la información del archivo json de metadata de videojuegos:

In [3]:
games_metadata = pd.read_json('games_metadata.json', lines=True)
games_metadata.head(5)

,app_id,description,tags
0,13500,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
1,22364,,[Action]
2,113020,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
3,226560,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."


Dado que el único feedback explícito que se posee en los datasets de interacciones es la columna "is_recommended", se añade una columna "rating" cuyo valor es 1 si "is_recommended" es "True", y su valor es 0 si "is_recommended" es "False". Esto se hace para cada uno de los df:

In [4]:
regla_rating = {True: 1, False: 0}

df_train['rating'] = df_train['is_recommended'].map(regla_rating)
df_test['rating'] = df_test['is_recommended'].map(regla_rating)
df_val['rating'] = df_val['is_recommended'].map(regla_rating)

El resultado es:

In [5]:
df_train.head(5)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id,rating
0,322330,0,0,2019-07-02,True,67.5,731,33484606,1
1,433340,0,0,2020-01-24,True,32.3,731,26236770,1
2,394360,2,0,2020-04-20,True,403.7,731,25992499,1
3,4700,0,0,2020-04-21,True,683.5,731,9845461,1
4,246090,3,0,2015-02-25,False,7.9,3128,28148695,0


## 2- Definición del modelo, formateo de datos, y entrenamiento

Se formatean los datos para la librería RecBole, se define el modelo, y se entrena:

In [6]:
import os
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

#carpeta del dataset
dataset_name = "dataset_recbole"
dataset_dir = f"./dataset_recbole"
os.makedirs(dataset_dir, exist_ok=True)

#se formatean los dataframes para que tengan las columnas requeridas por RecBole
df_train_lightgcn = df_train[["user_id", "app_id", "rating"]].copy()
df_test_lightgcn = df_test[["user_id", "app_id", "rating"]].copy()
df_val_lightgcn = df_val[["user_id", "app_id", "rating"]].copy()
df_train_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)
df_test_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)
df_val_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)

#se guarda cada split como archivo .inter en la carpeta definida
df_train_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.train.inter", sep="\t", index=False)
df_test_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.test.inter", sep="\t", index=False)
df_val_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.valid.inter", sep="\t", index=False)

#diccionario de configuración requerido por RecBole para entrenar el modelo:
config_dict = {
    "data_path": ".",
    "model": "LightGCN",
    "dataset": dataset_name,
    "format": "custom",
    "benchmark_filename" : ['train', 'valid', 'test'],
    "dataset_file": {
        "inter": {
            "train": f"{dataset_dir}/{dataset_name}.train.inter",
            "valid": f"{dataset_dir}/{dataset_name}.valid.inter",
            "test":  f"{dataset_dir}/{dataset_name}.test.inter"
        }
    },
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},
    "field_type": {
        "user_id": "token",
        "item_id": "token",
        "rating": "float"
    },
    "eval_args": {"mode": "full"},
    "epochs": 10,
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": 64,
    "show_progress": True,
    "learning_rate": 0.001,
    "reg_weight": 1e-5,
    "n_layers": 3,
    "topk": [10],
    "device": "cpu",
    #umbral de rating para considerar un ítem como relevante.
    #en este caso, todo ítem con rating >= 0.5 es considerado relevante.
    "threshold": {"rating": 0.5} 
}

#se crea el objeto Config de RecBole que guarda toda la configuración
# requerida por la librería, se carga el dataset, y se preparan los datos:

config = Config(config_dict=config_dict)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

#modelo LightGCN y el Trainer de RecBole:
model = LightGCN(config, train_data.dataset).to(config['device'])
trainer = Trainer(config, model)

#entrenamiento
trainer.fit(train_data, valid_data)
print()
print("Entrenamiento finalizado")

c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm


Entrenamiento finalizado


## 3- Generación de recomendaciones

Ahora, a partir del entrenamiento anterior se obtienen las listas de recomendación top 10 para cada usuario. Se comienza usando las ids internas de la librería, para luego convertirlas a las originales del dataset:

In [7]:
#se guardan las ids internas de los usuarios presentes en el dataset,
# evitando la id = 0 que se asocia al ['PAD'] que usa internamente RecBole
dataset_obj = test_data.dataset
uid_series = [uid for uid in range(1, dataset_obj.user_num) 
              if test_data.uid2history_item[uid] is not None]

#se obtienen las listas de recomendación top 10 para cada usuario
topk = 10
topk_result = full_sort_topk(uid_series, model, test_data, k=topk, device=config['device'])

Se guardan en un diccionario las recomendaciones con las ids de usuario e ítem originales, convirtiendo primero las ids internas de RecBole a las originales del dataset. El diccionario tiene como llaves la id del usuario y su valor es una lista con las 10 id de ítems recomendados: 

In [8]:
#se convierten las ids de usuarios internas de RecBole a las originales del dataset
user_original_ids = dataset_obj.id2token('user_id', uid_series)

#se convierten las ids de ítems internas de RecBole a las originales del dataset
topk_items_original = [
    dataset_obj.id2token('item_id', topk_result.indices[i])
    for i in range(len(uid_series))
]

#diccionario final de recomendaciones:

#con ids como strings:
# recommendations_dict = {int(user_id): items for user_id, items in zip(user_original_ids, topk_items_original)}

#con ids como enteros:
top10_lightgcn = {int(user_id): [int(item) for item in items] for user_id, items in zip(user_original_ids, topk_items_original)}


print("Ejemplo de 5 usuarios y sus recomendaciones:")
for user_id, items in list(top10_lightgcn.items())[:5]:
    print(f"Usuario {user_id} → {items}")

Ejemplo de 5 usuarios y sus recomendaciones:
Usuario 731 → [440, 304930, 374320, 444090, 1091500, 377160, 359550, 945360, 550, 292030]
Usuario 3128 → [440, 374320, 304930, 444090, 377160, 359550, 275850, 945360, 431960, 292030]
Usuario 4232 → [440, 304930, 444090, 374320, 377160, 1091500, 359550, 550, 292030, 431960]
Usuario 8297 → [440, 374320, 304930, 444090, 1091500, 377160, 359550, 945360, 550, 275850]
Usuario 9905 → [304930, 444090, 374320, 1091500, 377160, 359550, 945360, 550, 292030, 275850]


Usuarios con recomendaciones (en total el dataset tiene 3000):

In [9]:
len(top10_lightgcn)

9906

## 4- Métricas

La ejecución del modelo LightGCN de la librería RecBole entrega de por sí ciertas métricas: Recall@K, Precision@K, MRR@K, NDCG@K, y HitScore@K. Estas métricas se obtienen primero del objeto Trainer de la librería y se calculan sobre la data de testeo. Se obtiene un tipo OrderedDict que se convierte a diccionario:

In [10]:
test_result = trainer.evaluate(test_data)
metricas_finales = dict(test_result)
print(metricas_finales)

c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\trainer\trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.loa

{'recall@10': 0.0373, 'mrr@10': 0.0129, 'ndcg@10': 0.0175, 'hit@10': 0.045, 'precision@10': 0.0045}


Ahora, de las anteriores se guardan en variables sólo las métricas relevantes (las que se utilizan en este proyecto, todas excepto MRR@K) para más adelante mostrarlas en un resumen:

In [11]:
recall_lightgcn = metricas_finales['recall@10']
precision_lightgcn = metricas_finales['precision@10']
ndcg_lightgcn = metricas_finales['ndcg@10']
hitscore_lightgcn = metricas_finales['hit@10']

print(f"Recall@10: {recall_lightgcn}")
print(f"Precision@10: {precision_lightgcn}")
print(f"NDCG@10: {ndcg_lightgcn}")
print(f"HitScore@10: {hitscore_lightgcn}")

Recall@10: 0.0373
Precision@10: 0.0045
NDCG@10: 0.0175
HitScore@10: 0.045


Otra métrica que se utilizará es el F1 Score. Se calcula y guarda en una variable:

In [12]:
f1score_lightgcn = 2 * (precision_lightgcn * recall_lightgcn) / (precision_lightgcn + recall_lightgcn)
print(f"F1 Score@10: {f1score_lightgcn:.4f}")

F1 Score@10: 0.0080


Ahora, se calcula el MAP@K. Se comienzan por definir dos funciones base, una para calcular el Average Precision at K (AP@K) para un usuario y luego otra para calcular el MAP@K como el promedio del AP@K de todos los usuarios. Finalmente, se calcula el MAP@10:

In [15]:
def ap_at_k(items_relevantes, items_recomendados, k):
    if len(items_recomendados) > k:
        items_recomendados = items_recomendados[:k]
        
    score = 0
    num_hits = 0
    for i, p in enumerate(items_recomendados):
        if p in items_relevantes and p not in items_recomendados[:i]:
            num_hits += 1
            score += (num_hits / (i + 1))
    
    if not items_relevantes:
        return 0

    return score / min(len(items_relevantes), k)

def map_at_k(df_interacciones, dict_recomendados, k, umbral_relevante):
    ap_scores = []
    for user_id, lista_recomendados in dict_recomendados.items():
        items_relevantes = set(df_interacciones[(df_interacciones["user_id"] == user_id) & (df_interacciones["rating"] >= umbral_relevante)]["app_id"])
        ap = ap_at_k(items_relevantes, lista_recomendados, k)
        ap_scores.append(ap)
    
    return sum(ap_scores) / len(ap_scores)

#dataframe que une todas las interacciones (de los 3 dataframes del split):
df_final = pd.concat([df_train, df_val, df_test], ignore_index=True)

map_lightgcn = map_at_k(df_final, top10_lightgcn, k=10, umbral_relevante=0.5)
print(f"MAP@10: {map_lightgcn:.4f}")

MAP@10: 0.0018


Ahora, se calcula la diversidad promedio de las recomendaciones (Diversity). Para esto se analizan los géneros de videojuegos recomendados y la métrica representa cuántos géneros distintos de videojuegos se recomiendan en promedio. Se comienza definiendo dos funciones para realizar los cálculos y luego se obtiene Diversity: 

In [17]:
#código basado en el elaborado por Nicolás Bueno y Felipe Fuentes en Tarea del curso.

#función para calcular la cantidad de géneros distintos de videojuegos en la lista de recomendación de un usuario
def diversity_user(items_recomendados, dict_item_genero):
    if not items_recomendados:
        return 0
    
    unique_generos = set()
    for item_id in items_recomendados:
        if item_id in dict_item_genero:
            #se recorre la lista de tags (géneros) asociados al videojuego
            for tag in dict_item_genero[item_id]:
                unique_generos.add(tag)

    return float(len(unique_generos))

#función para calcular la cantidad promedio de géneros distintos de videojuegos en todas las listas de recomendación generadas
def diversity(recomendaciones, dict_item_genero):
    total = 0
    cant_usuarios = 0
    for recs in recomendaciones.values():
        total += diversity_user(recs, dict_item_genero)
        cant_usuarios += 1
        
    return total / max(cant_usuarios, 1)

#los géneros de un videojuego se consideran como los "tags" asociados en el archivo de metadata (guardado en en el dataframe games_metadata)
dict_item_genero = dict(zip(games_metadata["app_id"].astype(int), games_metadata["tags"]))
diversity_lightgcn = diversity(top10_lightgcn, dict_item_genero)
print(f"Diversidad promedio: {diversity_lightgcn}")

Diversidad promedio: 20.32404603270745


En resumen, las métricas calculadas del modelo son:

In [18]:
print("Resumen de las métricas:")
print()
print(f"Recall@10: {recall_lightgcn:.4f}")
print(f"Precision@10: {precision_lightgcn:.4f}")
print(f"F1 Score@10: {f1score_lightgcn:.4f}")
print(f"NDCG@10: {ndcg_lightgcn:.4f}")
print(f"HitScore@10: {hitscore_lightgcn:.4f}")
print(f"MAP@10: {map_lightgcn:.4f}")
print(f"Diversity: {diversity_lightgcn:.4f}")

Resumen de las métricas:

Recall@10: 0.0373
Precision@10: 0.0045
F1 Score@10: 0.0080
NDCG@10: 0.0175
HitScore@10: 0.0450
MAP@10: 0.0018
Diversity: 20.3240


## 5- Referencias

[1] Deng, K., He, X., Li, Y., Wang, M., Wang, X., & Zhang, Y. (2020). LightGCN: Simplifying and powering graph Convolution Network for recommendation. ArXiv. Obtenido de http://arxiv.org/abs/2002.02126

[2] Diapositivas de la clase "Clase de Evaluación: metricas de error y ranking", como apoyo para implementar cálculo de métricas. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/clases/s3_c1-metricas_v3.pdf

[3] Documentación de RecBole 1.2.1: https://recbole.io/docs/

[4] Documentación modelo LightGCN de Recbole: https://recbole.io/docs/user_guide/model/general/lightgcn.html

[5] Repositorio Recbole en GitHub: https://github.com/RUCAIBox/RecBole

[6] Práctico de métricas del curso del semestre pasado, utilizado como base principalmente para programar las métrica de Diversity. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/practicos/pr%C3%A1ctico_m%C3%A9tricas.ipynb

[7] Uso de IA. Se consultó a ChatGPT sobre la librería RecBole, el uso de funciones y formatos de LightGCN, y errores. Link al chat: https://chatgpt.com/share/68ffb7df-f778-8008-b287-14268d81d203